# Network hydraulics export for 1D particle tracking

This notebook is an add-on to `02_prms_legacy_models.ipynb`. It runs the same NHM configuration on the Delaware River Basin (DRB) with two changes:

1. A more realistic per-segment **hydraulic geometry**. The DRB parameter file carries bankfull width (`seg_width`) and depth (`seg_depth`), originally set from the Bieger et al. (2015) regional curves, but its width power law is nearly flat (`width_m = 0.015`) and it has no depth power law at all, so PRMS falls back to a single default depth relation for every segment. Here we anchor at-a-station power laws through each segment's bankfull point using Manning's equation and the Leopold and Maddock (1953) exponents (width 0.26, depth 0.40, velocity 0.34).
2. An **export** of the routed flows and their hydraulic geometry to one model-agnostic NetCDF file, the interface to the 1D river-network particle tracker in the `fluvial-particle` package.

## Prerequisites
As for the other example notebooks. This notebook regenerates the CBH NetCDF inputs it needs into its own output directory, so it does not depend on `02` having been run.

In [ ]:
import pathlib as pl
from shutil import rmtree

import matplotlib.pyplot as plt
import numpy as np
import pywatershed as pws
import xarray as xr
from pywatershed.utils import gis_files


class SegArrays(dict):
    """Per-segment arrays that ProcessPlot.plot_seg_var can plot."""

    def __init__(self, params, **arrays):
        super().__init__(arrays)
        self._params = params


gis_files.download()  # GIS files for plotting and for the export

domain_dir = pws.constants.__pywatershed_root__ / "data/drb_2yr"
nb_output_dir = pl.Path("./02a_network_hydraulics_export")
nb_output_dir.mkdir(exist_ok=True)

## Preprocess CBH files to NetCDF
Identical to notebook `02`.

In [ ]:
cbh_nc_dir = nb_output_dir / "drb_2yr_cbh_files"
if cbh_nc_dir.exists():
    rmtree(cbh_nc_dir)
cbh_nc_dir.mkdir(parents=True)

renamer = {
    "prcp": "prcp",
    "tmax": "tmax",
    "tmin": "tmin",
    "rhavg": "humidity_hru",
}
params = pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")
for stem in renamer:
    pws.utils.cbh_file_to_netcdf(
        domain_dir / f"{stem}.cbh",
        params,
        cbh_nc_dir / f"{renamer[stem]}.nc",
        rename_vars=renamer,
    )

## Derive at-a-station hydraulic geometry from bankfull parameters

`at_a_station_hydraulic_geometry` computes each segment's bankfull discharge with Manning's equation for a rectangular section of `seg_width` by `seg_depth` on `seg_slope` with roughness `mann_n`, then sets `width_alpha`, `width_m`, `depth_alpha`, `depth_m` so that `alpha * Q**m` returns the bankfull width and depth at that discharge. It returns a new `Parameters` object; the original is untouched. With `return_bankfull=True` we also get the bankfull discharge and velocity for plotting.

In [ ]:
params_hg, bankfull = pws.utils.at_a_station_hydraulic_geometry(
    params, return_bankfull=True
)
print("velocity exponent:", bankfull["velocity_exp"])
for name in ["width_alpha", "width_m", "depth_alpha", "depth_m"]:
    vals = params_hg.parameters[name]
    print(f"{name:12s} min={vals.min():.4g} median={np.median(vals):.4g} max={vals.max():.4g}")

In [ ]:
gis_dir = pws.utils.get_gis_dir("drb_2yr")
proc_plot = pws.analysis.ProcessPlot(gis_dir)

# ProcessPlot.plot_seg_var reads process[var_name] and
# process._params.coords["nhm_seg"], so SegArrays (a dict that also carries
# the parameters) stands in for a Process.
bankfull_arrays = SegArrays(
    params_hg,
    bankfull_flow=bankfull["bankfull_flow"],
    bankfull_velocity=bankfull["bankfull_velocity"],
)
proc_plot.plot_seg_var(
    "bankfull_flow",
    bankfull_arrays,
    title="Manning bankfull discharge (m3/s)",
    value_transform=np.log10,
)
proc_plot.plot_seg_var(
    "bankfull_velocity", bankfull_arrays, title="Manning bankfull velocity (m/s)"
)

## Run the NHM with the full hydraulic geometry process

The process list is the one from notebook `02` with `PRMSHydraulicGeometryFull` in place of `PRMSHydraulicGeometryWidthOnly`, so both depth and width come from the derived power laws. We add the geometry variables not already in the control's output list so the exporter can read them, and run the same six months as `02`.

In [ ]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
    pws.PRMSHydraulicGeometryFull,
    pws.PRMSStreamTempHumidityCBH,
]

control = pws.Control.load_prms(
    domain_dir / "nhm_stream_temp.control", warn_unused_options=False
)
control.edit_end_time(np.datetime64("1979-07-01T00:00:00"))
run_dir = nb_output_dir / "nhm"
if run_dir.exists():
    rmtree(run_dir)
control.options["netcdf_output_var_names"] += [
    "seg_flow_width",
    "seg_flow_depth",
    "seg_flow_area",
    "seg_flow_velocity",
    "seg_res_time",
]
control.options = control.options | {
    "input_dir": cbh_nc_dir,
    "imbalance_behavior": "warn",
    "calc_method": "numba",
    "netcdf_output_dir": run_dir,
}

In [ ]:
%%time
nhm = pws.Model(nhm_processes, control=control, parameters=params_hg)
nhm.run(finalize=True)

## What changed relative to the PRMS-default geometry?

Notebook `02` used the width-only process: the parameter file's nearly flat width power law (`width_m = 0.015`) and the PRMS default depth relation (`0.27 * Q**0.39`) for every segment. We recompute that default geometry from the routed flow and compare last-day velocity and depth with the new parameterization. Flows are identical in both cases because the geometry does not feed back on routing.

In [ ]:
from pywatershed.hydrology.prms_hydraulic_geometry import CFS_TO_CMS

q_last = nhm.processes["PRMSChannel"]["seg_outflow"] * CFS_TO_CMS
w_default = params.parameters["width_alpha"] * q_last ** params.parameters["width_m"]
d_default = 0.27 * q_last**0.39
v_default = np.where(w_default * d_default > 1e-6, q_last / (w_default * d_default), 0.0)

hg = nhm.processes["PRMSHydraulicGeometryFull"]
compare = SegArrays(
    params_hg,
    velocity_default=v_default,
    velocity_at_a_station=hg["seg_flow_velocity"],
    depth_default=d_default,
    depth_at_a_station=hg["seg_flow_depth"],
)
for name in compare:
    proc_plot.plot_seg_var(name, compare, title=f"{name} (last day)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].scatter(v_default, hg["seg_flow_velocity"], s=8)
ax[0].plot([0, v_default.max()], [0, v_default.max()], "k--", lw=1)
ax[0].set_xlabel("PRMS default velocity (m/s)")
ax[0].set_ylabel("at-a-station velocity (m/s)")
ax[1].scatter(d_default, hg["seg_flow_depth"], s=8)
ax[1].plot([0, d_default.max()], [0, d_default.max()], "k--", lw=1)
ax[1].set_xlabel("PRMS default depth (m)")
ax[1].set_ylabel("at-a-station depth (m)")
fig.suptitle("Last-day hydraulic geometry, default vs bankfull-anchored")
plt.show()

## Export network hydraulics for particle tracking

`export_network_hydraulics` reads the run directory and the parameters, adds the segment polylines from the GIS shapefile, and writes one NetCDF with:

- static per-reach fields: identifiers, downstream index (`to_index`, -1 at outlets), length, slope, Manning's n, midpoint elevation, bankfull width and depth;
- a `vertex` block of polyline coordinates with cumulative arc length per vertex, oriented upstream to downstream;
- per-reach daily time series in SI units: `flow_in`, `flow_out`, `velocity`, `depth`, `width`, `ustar` (shear velocity), `residence_time`, and (when the run wrote `seg_tave_water`) `water_temperature`.

This file is the interface to the 1D network particle tracker in `fluvial-particle`. A particle there is a reach index plus a distance from the reach's upstream end; when it passes the reach length it moves to `to_index`, and it exits at an outlet. Map positions come from scaling that distance onto the polyline's `vertex_dist`.

In [ ]:
network_file = nb_output_dir / "drb_network_hydraulics.nc"
pws.utils.export_network_hydraulics(
    params_hg,
    run_dir,
    network_file,
    segment_shp_file=gis_dir / "Segments_subset.shp",
)
network = xr.open_dataset(network_file)
network

In [ ]:
last = network.isel(time=-1)
from_file = SegArrays(
    params_hg,
    velocity=last["velocity"].values,
    depth=last["depth"].values,
    ustar=last["ustar"].values,
    residence_time_hours=last["residence_time"].values / 3600.0,
)
for name in from_file:
    proc_plot.plot_seg_var(name, from_file, title=f"{name} from the export (last day)")

print("outlets:", int(network["is_outlet"].sum()))
print("unconnected polylines:", network.attrs["n_unconnected"])
print(
    "median residence time (h):",
    float(np.nanmedian(network["residence_time"].where(network["flow_out"] > 0)) / 3600.0),
)
network.close()

## References
* Bieger, K., Rathjens, H., Allen, P.M., Arnold, J.G. (2015). Development and evaluation of bankfull hydraulic geometry relationships for the physiographic regions of the United States. JAWRA 51(3), 842-858.
* Leopold, L.B., Maddock, T. (1953). The hydraulic geometry of stream channels and some physiographic implications. USGS Professional Paper 252.
* Regan, R.S., Markstrom, S.L., LaFontaine, J.H., Norton, P.A., 2022, PRMS version 5.2.1: Precipitation-Runoff Modeling System (PRMS): U.S. Geological Survey Software Release, 02/10/2022.